# Summary

Create text files from previously crawls.

In [1]:
import os, sys
import re
import json

import pandas as pd
from tqdm import tqdm

utils_path = "../../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()


## Get a list of files in the GCP bucket

In [2]:
# Get a list of contents in a GCP bucket
gcs_bucket_name = "ccc-crawl_data_xb"
bcontents = gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                gcs_bucket_name=gcs_bucket_name,
                                path="")

# Pull out the csv files
cfiles = []
for bc in bcontents:
    fp = os.path.split(bc)
    ps = fp[0].split("/")

    source = ps[1] if len(ps) > 1 else ""
    file_name = fp[1] if fp[1].find(".csv") >= 0 else ""

    if len(file_name) > 0:
        cfiles.append(dict(source=source,
                           file_name=file_name,
                           path=fp[0]))

# Create a dataframe with a list of files and their paths
dfc = pd.DataFrame(data=cfiles)
dfc.head()


,source,file_name,path
0,aacc,wwwaaccncheedu_2025May01_1.csv,crawl_data/aacc/webpages_pdfs
1,aacc,wwwaaccncheedu_2025May01_2.csv,crawl_data/aacc/webpages_pdfs
2,aacc,wwwaaccncheedu_2025May01_3.csv,crawl_data/aacc/webpages_pdfs
3,aacc,wwwaaccncheedu_2025May01_4.csv,crawl_data/aacc/webpages_pdfs
4,aacc,wwwaaccncheedu_2025May01_5.csv,crawl_data/aacc/webpages_pdfs


## Create text files

In [4]:
def clean_ptag_texts(ptag_texts: list):
    '''
    Method to clean the ptag text. This method takes a list of ptag texts and
    returns a single string of cleaned ptag texts joined together.
    '''

    # remove unwanted characters
    pats = [r"\n|\xa0", r"\s+", r"\[\d+]",  r"\[…]", r"\s{2,}", r"\||>|/"]

    ptag_text = " ".join(ptag_texts)
    for pat in pats:
        ptag_text = re.sub(pat, " ", ptag_text)

    ptag_text = ptag_text.strip()

    return ptag_text

def create_json_lines_file(data, filename):
    """
    Creates a JSON Lines file from a list of Python objects.

    Args:
        data: A list of Python objects to be written to the file.
        filename: The name of the file to be created.
    """
    with open(filename, 'w') as outfile:
        for item in data:
            json_string = json.dumps(item)
            outfile.write(json_string + '\n')

local_output_path = "../data/jsonl_files"

sources = ['aacc', 'calmatters', 'cccaoe', 'cccco', 'ccleague', 'ccreview', 'columbia',
           'ecs', 'ican', 'lao', 'nsc', 'wikipedia']

################### Handle ccreview
sources = ['aacc', 'calmatters', 'cccaoe', 'cccco', 'ccleague', 'columbia', 'ecs', 'ican', 'lao',
           'nsc', 'wikipedia']

gcs_path = "crawl_data/jsonl_files"

# For each source
for source in tqdm(sources):

    # Filter GCP buc ket file listing to just this source
    mask = dfc["source"] == source
    idx0 = dfc[mask].index[0]
    fn_base = dfc.loc[idx0, "file_name"]
    path = dfc.loc[idx0, "path"]
    path = "{}/text_files".format(path[:path.find("/")])
    text_jsonl_filename = "{}_text.jsonl".format(fn_base[:fn_base.rfind("_")])

    # Get the crawl results for this source
    dfs = []
    for idx in dfc[mask].index:
        df = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                           gcs_bucket_name=gcs_bucket_name,
                                           gcs_directory=dfc.loc[idx, "path"],
                                           file_name=dfc.loc[idx, "file_name"],
                                           exact=False)
        dfs.append(df)

    # Create a single dataframe
    df = pd.concat(dfs)
    df = df.reset_index(drop=True)

    # Eliminate columns with too little text
    ########################### Handle case with divtag_text
    df["ptag_text_len"] = df["ptag_text"].str.len()

    text_len_min = 249
    mask = df["ptag_text_len"] > text_len_min
    df = df[mask]
    df = df.reset_index(drop=True)

    # Create a text string with all crawled text
    txt_lines = []
    for i, idx in enumerate(df.index):

        if "divtag_text" in df.columns:
            content=clean_ptag_texts(ptag_texts=[df.loc[idx, "ptag_text"], df.loc[idx, "divtag_text"]])
        else:
            content=clean_ptag_texts(ptag_texts=[df.loc[idx, "ptag_text"]])

        # txt_lines.append(dict(id="{}_{}".format(source, i),
        #                       title=df.loc[idx, "page_title"],
        #                       page_url=df.loc[idx, "page_url"],
        #                       crawl_time=df.loc[idx, "crawl_time"],
        #                       content=content)
        #                  )

        txt_lines.append(dict(id="{}_{}".format(source, i),
                              structData=dict(title=df.loc[idx, "page_title"],
                                              page_url=df.loc[idx, "page_url"],
                                              crawl_time=df.loc[idx, "crawl_time"],
                                              text="{}: {}".format(df.loc[idx, "page_url"], content)
                                              ),
                              content="text/html",
                              uri="gs://{}/{}/{}".format(gcs_bucket_name,
                                                         gcs_path,
                                                         text_jsonl_filename)
                              )
                 )

     # Save a local jsonl file
    create_json_lines_file(data=txt_lines,
                           filename=os.path.join(local_output_path, text_jsonl_filename))

    # Save descriptions dataframe in a CSV file on GCP
    # gct.save_file_to_bucket(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
    #                         gcs_bucket_name=gcs_bucket_name,
    #                         gcs_directory=path,
    #                         file_name=text_txt_filename,
    #                         content=crawled_txt)


100%|██████████| 11/11 [00:46<00:00,  4.25s/it]


## Upload JSONL files to GCP

In [5]:
gcs_path = "crawl_data/jsonl_files"

gct.upload_directory_to_gcs(local_directory=local_output_path,
                            gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                            gcs_bucket_name=gcs_bucket_name,
                            gcs_directory=gcs_path)



Uploaded ../data/jsonl_files/ccrctccolumbiaedu_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_filesccrctccolumbiaedu_2025May01_text.jsonl
Uploaded ../data/jsonl_files/laocagov_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_fileslaocagov_2025May01_text.jsonl
Uploaded ../data/jsonl_files/wwwecsorg_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_fileswwwecsorg_2025May01_text.jsonl
Uploaded ../data/jsonl_files/icangotocollegecom_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_filesicangotocollegecom_2025May01_text.jsonl
Uploaded ../data/jsonl_files/nscresearchcenterorg_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_filesnscresearchcenterorg_2025May01_text.jsonl
Uploaded ../data/jsonl_files/wwwccleagueorg_2025May01_text.jsonl to gs://ccc-crawl_data_xb/crawl_data/jsonl_fileswwwccleagueorg_2025May01_text.jsonl
Uploaded ../data/jsonl_files/calmattersdigitaldemocracyorg_2025May01_text.jsonl to gs://ccc-crawl_data

In [61]:
txt = "\n\n".join(txt_list)

txt


test = "“On behalf of the board […] Filed Under: News, Press Release, Press Releases, Uncategorized FOR IMMEDIATE RELEASE January 21, 2025 Contact: Martha M. Parham, Ed.D. 202-728-0200 x209 mparham@aacc.nche.edu Community College Innovation Challenge invites community college students to create STEM solutions to real-world problems Washington, D.C. – Today, the American Association of Community Colleges (AACC) in partnership with the National Science Foundation (NSF), has launched its platform to advance student […] Filed Under: News, Press Release, Press Releases July 22, 2024 For Immediate Release Media Contacts: All Within My Hands Foundation Renee Richardson, Director of Philanthropy renee@allwithinmyhands.org (415) 458-1532 x10 American Association of Community Colleges Dr. Martha M. Parham, Sr. Vice President, Public Relations mparham@aacc.nche.edu (202) 728-0200 Year 6 is the largest grant to date with $2.6M"

pat= r"\[…]"
ptag_text = re.sub(pat, " ", test)

ptag_text = clean_ptag_texts(ptag_texts=[test])
ptag_text

'“On behalf of the board Filed Under: News, Press Release, Press Releases, Uncategorized FOR IMMEDIATE RELEASE January 21, 2025 Contact: Martha M. Parham, Ed.D. 202-728-0200 x209 mparham@aacc.nche.edu Community College Innovation Challenge invites community college students to create STEM solutions to real-world problems Washington, D.C. – Today, the American Association of Community Colleges (AACC) in partnership with the National Science Foundation (NSF), has launched its platform to advance student Filed Under: News, Press Release, Press Releases July 22, 2024 For Immediate Release Media Contacts: All Within My Hands Foundation Renee Richardson, Director of Philanthropy renee@allwithinmyhands.org (415) 458-1532 x10 American Association of Community Colleges Dr. Martha M. Parham, Sr. Vice President, Public Relations mparham@aacc.nche.edu (202) 728-0200 Year 6 is the largest grant to date with $2.6M'

In [39]:
len(txt)

1349884